# QC for final assemblies

1.) Input QC
- PoreC
- ReadLengths UL
- ReadLengths HQ_herro

2.) Assembly QC

In [45]:
library(tidyverse)
library(ggplot2)
library(gt)

In [46]:
samples <- c(
    "GE-MED-T2T00",
    "GE-MED-T2T04"
)

## 1.) Input QC

In [47]:
source("../scripts/02_plot_read_stats.R")

types <- c("HQ_herro.50x", "UL.70x")

dt_input_qc <- expand.grid(sample = samples, type = types) %>%
    mutate(path = paste0("../../assembly/input_qc/", sample, "/", sample, ".", type, "/read_stats.txt"))

dt_input_qc

sample,type,path
<fct>,<fct>,<chr>
GE-MED-T2T00,HQ_herro.50x,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt
GE-MED-T2T04,HQ_herro.50x,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt
GE-MED-T2T00,UL.70x,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt
GE-MED-T2T04,UL.70x,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt


In [48]:
dt_l <- list()
for (i in 1:nrow(dt_input_qc)) {
    print(paste("Processing", dt_input_qc$sample[i]))
    dt_l[[dt_input_qc$path[i]]] <- process_sequencing_file(dt_input_qc$path[i])
} 

[1] "Processing GE-MED-T2T00"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in calculate_2d_density(result$plot_data):
“Skipping 2D density for sample GE-MED-T2T00_HQ_herro.50x due to insufficient variation”


[1] "Calculating read length histogram"
[1] "Processing GE-MED-T2T04"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in calculate_2d_density(result$plot_data):
“Skipping 2D density for sample GE-MED-T2T04_HQ_herro.50x due to insufficient variation”


[1] "Calculating read length histogram"
[1] "Processing GE-MED-T2T00"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in bkde2D(cbind(x_vals, y_vals), bandwidth = c(h1, h2), gridsize = c(n, :
“Binning grid too coarse for current (small) bandwidth: consider increasing 'gridsize'”


[1] "Calculating read length histogram"
[1] "Processing GE-MED-T2T04"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in bkde2D(cbind(x_vals, y_vals), bandwidth = c(h1, h2), gridsize = c(n, :
“Binning grid too coarse for current (small) bandwidth: consider increasing 'gridsize'”


[1] "Calculating read length histogram"


In [49]:
for (i in 1:length(dt_l)) {
    dt_l[[i]]$summary_stats$path <- names(dt_l)[i]
    dt_l[[i]]$read_length_density$path <- names(dt_l)[i]
    dt_l[[i]]$quality_density$path <- names(dt_l)[i]
    dt_l[[i]]$density_2d$path <- names(dt_l)[i]
}
dt_stats <- bind_rows(lapply(dt_l, function(x) x$summary_stats)) %>%
    inner_join(dt_input_qc, by = c("path" = "path")) %>%
    select(c(-sample_name))
dt_read_length_density <- bind_rows(lapply(dt_l, function(x) x$read_length_density)) %>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_quality_density <- bind_rows(lapply(dt_l, function(x) x$quality_density))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_density_2d <- bind_rows(lapply(dt_l, function(x) x$density_2d))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))


In [50]:
dt_stats

N50,mean_quality,median_quality,total_bases,num_reads,mean_read_length,median_read_length,yield_above_80kb,yield_above_100kb,yield_above_200kb,yield_above_500kb,reads_above_1MB,path,sample,type
<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<fct>,<fct>
80466,33.00000,33.00000,160000022814,2066094,77440.82,64294,80618140989,56517647266,9702465035,30196590,0,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt,GE-MED-T2T00,HQ_herro.50x
110550,33.00000,33.00000,160000007054,1495219,107007.74,89944,120953482092,92183929678,23968796860,658038978,0,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt,GE-MED-T2T04,HQ_herro.50x
122012,24.98229,24.98229,89135967621,720989,123630.14,108256,89134847621,63954213335,12712024242,537756923,75,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt,GE-MED-T2T00,UL.70x
137001,24.77841,24.77841,137813393040,1014245,135877.81,115656,137811793040,108199024306,32430409794,1509263333,67,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt,GE-MED-T2T04,UL.70x


In [51]:
dt_stats_reduced <- dt_stats %>%
    select(sample, type, total_bases, N50, yield_above_100kb, median_quality, reads_above_1MB)
dt_stats_reduced

sample,type,total_bases,N50,yield_above_100kb,median_quality,reads_above_1MB
<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
GE-MED-T2T00,HQ_herro.50x,160000022814,80466,56517647266,33.00000,0
GE-MED-T2T04,HQ_herro.50x,160000007054,110550,92183929678,33.00000,0
GE-MED-T2T00,UL.70x,89135967621,122012,63954213335,24.98229,75
GE-MED-T2T04,UL.70x,137813393040,137001,108199024306,24.77841,67


### PoreC QC, using results from wf-porec pipeline

In [52]:
#todo

## Assembly QC Results

In [64]:
source("../scripts/13_process_assembly_qc.R")
dt <- read_tsv("/mnt/storage3b/projects/no_ngsd/ahthapp1_T2T_ONT/assembly/qc/qc_samples.tsv") %>%
    process_qc_table %>%
    mutate(haplotype = ifelse(is.na(haplotype), "both", haplotype))
head(dt)

Rows: 2254 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (6): metric, haplotype, asm_method, asm_name, source, chromosome
dbl (1): value

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


metric,value,haplotype,asm_method,asm_name,source,chromosome
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
Assembly Length,2.926889e+09,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA
% of Ref covered by Assembly,9.031000e-01,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA
% of Ref duplicated in Assembly,8.900000e-03,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA
% of Assembly covered by Ref,9.698000e-01,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA
NG75,9.938695e+07,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA
NG50,1.388843e+08,haplotype1,phased_verkko,GE-MED-T2T00,asmstat,NA


In [65]:
dt_qc_table <- dt %>%
  filter() %>%
  mutate(haplotype = ifelse(haplotype == "assembly", "both", haplotype)) %>%
  group_by(asm_name, metric, haplotype) %>%
  summarise(value = sum(value, na.rm = TRUE), .groups = 'drop') %>%
  mutate(
    formatted_value = case_when(

      metric %in% c("Error Rate") ~ paste0(round(value * 100, 3), "%"),
      TRUE ~ as.character(round(value, 2))
    )
  ) %>%
  select(-value) %>%  # Remove original value column
  pivot_wider(
    names_from = metric, 
    values_from = formatted_value
  )

write_tsv(dt_qc_table, "../../doc/tables/qc_samples_table.tsv")

dt_qc_table  %>%
  knitr::kable()



|asm_name     |haplotype  |% of Assembly covered by Ref |% of Ref covered by Assembly |% of Ref duplicated in Assembly |Assembly Length |Average Ungapped Alignment length |Covered Variants (Total) |Error Rate |Genome Completeness |Missing Multi-Copy Genes (%) |NG50      |NG75      |NGA50   |Number of Breaks in Assembly |Number of T2T chromosomes |Overall Switch Flip Rate (%) |Overall Switch Rate (%) |Phase Blocks |Quality Value |Variants |Variants Phased |Variants Unphased |avg_gap_size |gap_n50 |max_gap_size |min_gap_size |n_contigs |n_contigs_over_10mb |singletons |total_gaps |total_n_count |
|:------------|:----------|:----------------------------|:----------------------------|:-------------------------------|:---------------|:---------------------------------|:------------------------|:----------|:-------------------|:----------------------------|:---------|:---------|:-------|:----------------------------|:-------------------------|:----------------------------|:----------------

In [70]:
# Select and prepare data
qc_selected <- dt_qc_table %>%
  select(
    asm_name,
    haplotype,
    `Assembly Length`,
    n_contigs_over_10mb,
    n_contigs,
    `% of Assembly covered by Ref`,
    `% of Ref covered by Assembly`,
    `Genome Completeness`, # This seems to be a direct percentage value (e.g. 94.62 means 94.62%)
    `Missing Multi-Copy Genes (%)`,
    `Quality Value`, # This is likely a Phred-like score, not percentage
    `Overall Switch Rate (%)`, # This is a proportion (0.05 = 5%)
    `Number of T2T chromosomes`,
    total_n_count,
    total_gaps
  ) %>%
  mutate(
    # Clean column names for easier use in gt (optional but good practice)
    # Here we will use backticks later, so not strictly necessary for this script
    
    # Ensure other numeric columns are numeric (read_tsv is good, but explicit is safer)
    # Note: `Overall Switch Rate (%)` is already a proportion like 0.05
    # `Genome Completeness` values like 94.62 are treated as direct percentage values
    # `Quality Value` is a score
    across(c(`Assembly Length`, `Quality Value`, `Overall Switch Rate (%)`,`% of Assembly covered by Ref`, `% of Ref covered by Assembly`, `Genome Completeness`, total_n_count), as.numeric),
    across(c( `Number of T2T chromosomes`), as.integer),
    
    # Factor haplotype for desired order in table rows
    haplotype = factor(haplotype, levels = c("haplotype1", "haplotype2", "both"))
  ) %>%
  arrange(asm_name, haplotype)



In [75]:
# Create the gt table
qc_gt_table <- qc_selected %>% 
  gt(rowname_col = "haplotype", groupname_col = "asm_name") %>%
  tab_header(
    title = md("**Assembly QC Metrics**"),
    subtitle = "Selected key metrics"
  ) %>%
  cols_label(
    `Assembly Length` = html("Assembly<br>Length"),
    `Genome Completeness` = html("Genome<br>Completeness"),
    `Overall Switch Rate (%)` = html("Switch<br>Error Rate"),
    `Number of T2T chromosomes` = html("T2T<br>Chroms"),
    `total_n_count` = html("Total Ns"),
    `total_gaps` = html("Total Gaps"),
    `n_contigs`="n Contigs",
    `n_contigs_over_10mb`="n Contigs >10Mbp",
  ) %>%
  # Formatting numeric columns
  fmt_number(
    columns = `Assembly Length`,
    decimals = 2,
    scale_by = 1/1e9, # Convert to Gbp
    pattern = "{x} Gbp"
  ) %>%
   fmt_number(
    columns = `Number of T2T chromosomes`,
    decimals = 0
  ) %>%
  fmt_number(
    columns = `% of Assembly covered by Ref`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
  fmt_number( 
    columns = `% of Ref covered by Assembly`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
   fmt_number( # For Genome Completeness (e.g., BUSCO score, already in percent value)
    columns = `Genome Completeness`,
    decimals = 2,
    pattern = "{x}%" # Append % sign
  ) %>%
  # Handle missing values
  fmt_missing(
    columns = everything(),
    missing_text = "—" # Display NAs as a dash
  ) %>%
  # Align columns
  cols_align(
    align = "center",
    columns = where(is.numeric) # Center numeric columns
  ) %>%
  cols_align(
    align = "left", # The rowname_col (haplotype) defaults to left, which is good
    columns = `asm_name` # The groupname_col (asm_name) also default to left.
  ) %>%
  # Add some styling
  tab_options(
    table.border.top.color = "black",
    table.border.bottom.color = "black",
    table.width = pct(90), # Make table width 90% of container
    heading.title.font.size = px(20),
    heading.subtitle.font.size = px(15),
    column_labels.border.bottom.color = "black",
    column_labels.font.weight = "bold",
    row_group.font.weight = "bold",
    row_group.background.color = "#f0f0f0", # Light grey for group headers
    table_body.hlines.color = "#D3D3D3" # Light grey horizontal lines in body
  ) %>%
  tab_source_note(
    source_note = "Data from qc_samples_table.txt. Gbp: Giga base pairs, Mbp: Mega base pairs."
  )

gts <- function(gt_table){
   gt:::as.tags.gt_tbl(gt_table)
}

# Print the table
qc_gt_table %>% gts

Shiny tags cannot be represented in plain text (need html)

### Table for Hifiasm

In [76]:
source("../scripts/13_process_assembly_qc.R")
dt_hifiasm <- read_tsv("/mnt/storage3b/projects/no_ngsd/ahthapp1_T2T_ONT/assembly/qc/qc_hifiasm.tsv") %>%
    process_qc_table %>%
    mutate(haplotype = ifelse(is.na(haplotype), "both", haplotype))
head(dt_hifiasm)

Rows: 2254 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (6): metric, haplotype, asm_method, asm_name, source, chromosome
dbl (1): value

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


metric,value,haplotype,asm_method,asm_name,source,chromosome
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
Assembly Length,3.038969e+09,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA
% of Ref covered by Assembly,9.394000e-01,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA
% of Ref duplicated in Assembly,8.500000e-03,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA
% of Assembly covered by Ref,9.722000e-01,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA
NG75,1.063044e+08,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA
NG50,1.379055e+08,haplotype1,phased_hifiasm,TUE_02_03_hifiasm_ont,asmstat,NA


In [77]:
dt_qc_table_hifiasm <- dt_hifiasm %>%
  filter() %>%
  mutate(haplotype = ifelse(haplotype == "assembly", "both", haplotype)) %>%
  group_by(asm_name, metric, haplotype) %>%
  summarise(value = sum(value, na.rm = TRUE), .groups = 'drop') %>%
  mutate(
    formatted_value = case_when(

      metric %in% c("Error Rate") ~ paste0(round(value * 100, 3), "%"),
      TRUE ~ as.character(round(value, 2))
    )
  ) %>%
  select(-value) %>%  # Remove original value column
  pivot_wider(
    names_from = metric, 
    values_from = formatted_value
  )

write_tsv(dt_qc_table_hifiasm, "../../doc/tables/qc_samples_table.tsv")

dt_qc_table_hifiasm  %>%
  knitr::kable()



|asm_name                |haplotype  |% of Assembly covered by Ref |% of Ref covered by Assembly |% of Ref duplicated in Assembly |Assembly Length |Average Ungapped Alignment length |Covered Variants (Total) |Error Rate |Genome Completeness |Missing Multi-Copy Genes (%) |NG50      |NG75      |NGA50   |Number of Breaks in Assembly |Number of T2T chromosomes |Overall Switch Flip Rate (%) |Overall Switch Rate (%) |Phase Blocks |Quality Value |Variants |Variants Phased |Variants Unphased |avg_gap_size |gap_n50 |max_gap_size |min_gap_size |n_contigs |n_contigs_over_10mb |singletons |total_gaps |total_n_count |
|:-----------------------|:----------|:----------------------------|:----------------------------|:-------------------------------|:---------------|:---------------------------------|:------------------------|:----------|:-------------------|:----------------------------|:---------|:---------|:-------|:----------------------------|:-------------------------|:------------------------

In [79]:
# Select and prepare data
qc_selected_hifiasm <- dt_qc_table_hifiasm %>%
  select(
    asm_name,
    haplotype,
    `Assembly Length`,
    n_contigs_over_10mb,
    n_contigs,
    `% of Assembly covered by Ref`,
    `% of Ref covered by Assembly`,
    `Genome Completeness`, # This seems to be a direct percentage value (e.g. 94.62 means 94.62%)
    `Missing Multi-Copy Genes (%)`,
    `Quality Value`, # This is likely a Phred-like score, not percentage
    `Overall Switch Rate (%)`, # This is a proportion (0.05 = 5%)
    `Number of T2T chromosomes`,
    total_n_count,
    total_gaps
  ) %>%
  mutate(
    # Clean column names for easier use in gt (optional but good practice)
    # Here we will use backticks later, so not strictly necessary for this script
    
    # Ensure other numeric columns are numeric (read_tsv is good, but explicit is safer)
    # Note: `Overall Switch Rate (%)` is already a proportion like 0.05
    # `Genome Completeness` values like 94.62 are treated as direct percentage values
    # `Quality Value` is a score
    across(c(`Assembly Length`, `Quality Value`, `Overall Switch Rate (%)`,`% of Assembly covered by Ref`, `% of Ref covered by Assembly`, `Genome Completeness`, total_n_count), as.numeric),
    across(c( `Number of T2T chromosomes`), as.integer),
    
    # Factor haplotype for desired order in table rows
    haplotype = factor(haplotype, levels = c("haplotype1", "haplotype2", "both"))
  ) %>%
  arrange(asm_name, haplotype)

In [81]:
# Create the gt table
qc_gt_table_hifiasm <- qc_selected_hifiasm %>% 
  gt(rowname_col = "haplotype", groupname_col = "asm_name") %>%
  tab_header(
    title = md("**Assembly QC Metrics**"),
    subtitle = "Selected key metrics"
  ) %>%
  cols_label(
    `Assembly Length` = html("Assembly<br>Length"),
    `Genome Completeness` = html("Genome<br>Completeness"),
    `Overall Switch Rate (%)` = html("Switch<br>Error Rate"),
    `Number of T2T chromosomes` = html("T2T<br>Chroms"),
    `total_n_count` = html("Total Ns"),
    `total_gaps` = html("Total Gaps"),
    `n_contigs`="n Contigs",
    `n_contigs_over_10mb`="n Contigs >10Mbp",
  ) %>%
  # Formatting numeric columns
  fmt_number(
    columns = `Assembly Length`,
    decimals = 2,
    scale_by = 1/1e9, # Convert to Gbp
    pattern = "{x} Gbp"
  ) %>%
   fmt_number(
    columns = `Number of T2T chromosomes`,
    decimals = 0
  ) %>%
  fmt_number(
    columns = `% of Assembly covered by Ref`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
  fmt_number( 
    columns = `% of Ref covered by Assembly`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
   fmt_number( # For Genome Completeness (e.g., BUSCO score, already in percent value)
    columns = `Genome Completeness`,
    decimals = 2,
    pattern = "{x}%" # Append % sign
  ) %>%
  # Handle missing values
  fmt_missing(
    columns = everything(),
    missing_text = "—" # Display NAs as a dash
  ) %>%
  # Align columns
  cols_align(
    align = "center",
    columns = where(is.numeric) # Center numeric columns
  ) %>%
  cols_align(
    align = "left", # The rowname_col (haplotype) defaults to left, which is good
    columns = `asm_name` # The groupname_col (asm_name) also default to left.
  ) %>%
  # Add some styling
  tab_options(
    table.border.top.color = "black",
    table.border.bottom.color = "black",
    table.width = pct(90), # Make table width 90% of container
    heading.title.font.size = px(20),
    heading.subtitle.font.size = px(15),
    column_labels.border.bottom.color = "black",
    column_labels.font.weight = "bold",
    row_group.font.weight = "bold",
    row_group.background.color = "#f0f0f0", # Light grey for group headers
    table_body.hlines.color = "#D3D3D3" # Light grey horizontal lines in body
  ) %>%
  tab_source_note(
    source_note = "Data from qc_samples_table.txt. Gbp: Giga base pairs, Mbp: Mega base pairs."
  )

gts <- function(gt_table){
   gt:::as.tags.gt_tbl(gt_table)
}

# Print the table
qc_gt_table_hifiasm %>% gts

Shiny tags cannot be represented in plain text (need html)